# Module 6: Project Data Cleaning

**Project:** Medi-Cal Provider Availability Review  
**Student:** Angana Shah

This notebook documents the cleaning and verification of the Medi-Cal provider dataset. The original data is kept unchanged, and the cleaning steps are also available in `cleaned_data.py`.

In [1]:
import pandas as pd

medical = pd.read_csv("../medical.csv", low_memory=False)

print("Dataset shape:", medical.shape)

Dataset shape: (372778, 46)


In [ ]:
# Inspect data types and missing values
print(medical.dtypes)
print("\nMissing values:")
print(medical.isna().sum())

In [ ]:
print("Exact duplicate rows:", medical.duplicated().sum())
print("Repeated Provider_Number values:",
      medical["Provider_Number"].duplicated().sum())
print("_id duplicates:", medical["_id"].duplicated().sum())
print("OBJECTID duplicates:", medical["OBJECTID"].duplicated().sum())

In [ ]:
# Convert enrollment date
date_test = pd.to_datetime(
    medical["Enroll_Status_Eff_DT"],
    errors="coerce"
)

print("Failed date conversions:", date_test.isna().sum())

medical["Enroll_Status_Eff_DT"] = date_test

# Convert NPI from numeric to identifier/string
npi_as_number = pd.to_numeric(
    medical["NPI"],
    errors="coerce"
)

failed_npi = (
    npi_as_number.isna().sum()
    - medical["NPI"].isna().sum()
)

medical["NPI"] = npi_as_number.astype("Int64").astype("string")

print("Failed NPI conversions:", failed_npi)
print("Date type:", medical["Enroll_Status_Eff_DT"].dtype)
print("NPI type:", medical["NPI"].dtype)

In [ ]:
# Find City values with extra spaces
city_with_spaces = (
    medical["City"].notna()
    & (medical["City"] != medical["City"].str.strip())
)

print("City values with extra spaces before cleaning:",
      city_with_spaces.sum())

# Save five original records for verification
check_ids = [28449, 65686, 74431, 80478, 107422]

original_checks = medical.loc[
    medical["_id"].isin(check_ids),
    ["_id", "City"]
].copy()

# Clean City
medical["City"] = medical["City"].str.strip()

# Check again
city_with_spaces_after = (
    medical["City"].notna()
    & (medical["City"] != medical["City"].str.strip())
)

print("City values with extra spaces after cleaning:",
      city_with_spaces_after.sum())

In [ ]:
print("Dates before 1900:",
      (medical["Enroll_Status_Eff_DT"] < "1900-01-01").sum())

print("Dates after today:",
      (medical["Enroll_Status_Eff_DT"] > pd.Timestamp.today()).sum())

print("Latitude = 0:", (medical["Latitude"] == 0).sum())
print("Longitude = 0:", (medical["Longitude"] == 0).sum())
print("Missing CountyName:", medical["CountyName"].isna().sum())

In [ ]:
record_checks = original_checks.copy()

record_checks["Expected"] = record_checks["City"].str.strip()

record_checks["Actual"] = record_checks["_id"].map(
    medical.set_index("_id")["City"]
)

record_checks["Match"] = (
    record_checks["Expected"] == record_checks["Actual"]
)

record_checks["Original"] = record_checks["City"].map(repr)
record_checks["Expected"] = record_checks["Expected"].map(repr)
record_checks["Actual"] = record_checks["Actual"].map(repr)

record_checks[
    ["_id", "Original", "Expected", "Actual", "Match"]
]

In [ ]:
# Save cleaned dataset
medical.to_csv("../medical_cleaned.csv", index=False)

# Create 50-row sample including difficult cases
difficult_cases = medical.loc[
    medical["_id"].isin(check_ids)
]

other_cases = medical.loc[
    ~medical["_id"].isin(check_ids)
].head(45)

sample = pd.concat(
    [difficult_cases, other_cases],
    ignore_index=True
)

sample.to_csv(
    "medical_cleaned_sample_50.csv",
    index=False
)

print("Final rows:", len(medical))
print("Sample rows:", len(sample))